# Notebook 05 — Double Descent & Interpolation
**DSC 240 · Machine Learning · UCSD**

> *"The best way to solve the problem from a practical standpoint is you build a very big system… basically you want to make sure you hit the zero training error."* — Ruslan Salakhutdinov, Simons Institute, 2017

Classical statistics says: find the **sweet spot** between underfitting and overfitting.  
Modern ML says: go **past** the interpolation threshold — the risk drops again.

This notebook reproduces the double descent phenomenon and explores why interpolation works.

Reference: Belkin, Hsu, Ma, Mandal (PNAS 2019)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lstsq

np.random.seed(0)
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'train': '#2D6A4F',
    'test': '#E63946',
    'bayes': '#F4A261',
    'interp': '#457B9D'
}

## 1. Setup: Random ReLU Features

We use a simple model from the slides:

$$h_d(x) = \sum_{j=1}^{d} \alpha_j \, \text{ReLU}(b_j^\top x + c_j)$$

- $(b_j, c_j)$ are **random and fixed** (random features)
- Only $\alpha_j$ are trained via **linear regression** on the random feature matrix
- Varying $d$ (number of features) from $d \ll n$ to $d \gg n$ traces the full double descent curve

In [ ]:
def relu(x):
    return np.maximum(x, 0)

def random_relu_features(X, d, seed=42):
    """Map X (n x p) to random ReLU feature matrix (n x d)."""
    rng = np.random.RandomState(seed)
    p = X.shape[1]
    B = rng.randn(p, d)   # weights
    c = rng.randn(d)       # biases
    return relu(X @ B + c)

def fit_and_evaluate(X_train, y_train, X_test, y_test, d, lam=0.0):
    """
    Fit linear model on random ReLU features of dimension d.
    Returns (train_mse, test_mse).
    lam: ridge regularization (0 = minimum-norm interpolation via lstsq)
    """
    Phi_train = random_relu_features(X_train, d)
    Phi_test  = random_relu_features(X_test, d)
    
    n = len(y_train)
    if lam > 0:
        # Ridge: (ΦᵀΦ + λI)α = Φᵀy
        A = Phi_train.T @ Phi_train + lam * np.eye(d)
        b = Phi_train.T @ y_train
        alpha = np.linalg.solve(A, b)
    else:
        # Minimum-norm least squares (interpolates when d > n)
        alpha, _, _, _ = lstsq(Phi_train, y_train)
    
    train_pred = Phi_train @ alpha
    test_pred  = Phi_test  @ alpha
    
    train_mse = np.mean((train_pred - y_train)**2)
    test_mse  = np.mean((test_pred  - y_test )**2)
    return train_mse, test_mse

## 2. Generate Regression Data with Noise

In [ ]:
# Regression: y = f*(x) + noise
n_train = 200
n_test  = 500
p_input = 10       # input dimension
sigma   = 0.3      # noise level

rng = np.random.RandomState(1)
w_true = rng.randn(p_input)
w_true /= np.linalg.norm(w_true)

X_train = rng.randn(n_train, p_input)
y_train = X_train @ w_true + sigma * rng.randn(n_train)

X_test  = rng.randn(n_test, p_input)
y_test  = X_test  @ w_true + sigma * rng.randn(n_test)

# Bayes-optimal (irreducible) error
bayes_error = sigma**2
print(f'Training samples: {n_train}')
print(f'Bayes (irreducible) error: {bayes_error:.4f}')

## 3. Sweep Model Complexity — Classical U then Double Descent

In [ ]:
# Feature dimensions to try — span from far underparameterized to far overparameterized
d_values = np.unique(np.round(
    np.concatenate([
        np.linspace(5, n_train - 5, 25),          # classical regime
        np.linspace(n_train + 5, 4*n_train, 20),  # interpolating regime
    ])
).astype(int))

train_errors, test_errors = [], []

for d in d_values:
    # Average over 5 random feature draws to reduce variance
    tr_list, te_list = [], []
    for seed in range(5):
        tr, te = fit_and_evaluate(X_train, y_train, X_test, y_test, d, lam=0.0)
        # Replace NaN/Inf with a large value (occurs right at d=n, singular)
        tr_list.append(min(tr, 5.0) if np.isfinite(tr) else 5.0)
        te_list.append(min(te, 5.0) if np.isfinite(te) else 5.0)
    train_errors.append(np.mean(tr_list))
    test_errors.append(np.mean(te_list))

train_errors = np.array(train_errors)
test_errors  = np.array(test_errors)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(d_values, train_errors, '-o', color=COLORS['train'], lw=2, ms=4, label='Train MSE', alpha=0.9)
ax.plot(d_values, test_errors, '-s', color=COLORS['test'], lw=2, ms=4, label='Test MSE', alpha=0.9)
ax.axhline(bayes_error, color=COLORS['bayes'], lw=1.5, ls='--', label=f'Bayes error (σ²={bayes_error})')
ax.axvline(n_train, color='gray', lw=1.5, ls=':', alpha=0.7, label=f'Interpolation threshold (d=n={n_train})')

# Annotate regimes
ax.text(n_train * 0.4, max(test_errors)*0.85, 'Classical\nRegime',
        ha='center', fontsize=11, color='#555', style='italic')
ax.text(n_train * 2.5, max(test_errors)*0.6, 'Modern\nInterpolating Regime',
        ha='center', fontsize=11, color='#555', style='italic')

ax.set_xlabel('Model complexity d (# random ReLU features)', fontsize=13)
ax.set_ylabel('Mean Squared Error', fontsize=13)
ax.set_title('Double Descent Risk Curve\n(Random ReLU Features + Minimum-Norm Interpolation)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(test_errors) * 1.15)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.savefig('../assets/figures/double_descent.png', dpi=130)
plt.show()

## 4. Ridge Regularization vs. Interpolation

Classical theory says: regularize. Modern theory says: interpolating solutions can be better. Let's compare directly.

In [ ]:
d_large = 1000  # deeply overparameterized
lambdas = np.logspace(-4, 2, 40)
ridge_test = []

for lam in lambdas:
    results = [fit_and_evaluate(X_train, y_train, X_test, y_test, d_large, lam=lam)[1] for s in range(5)]
    ridge_test.append(np.mean(results))

# Interpolation (λ=0) test error at d=1000
interp_test = np.mean([fit_and_evaluate(X_train, y_train, X_test, y_test, d_large, lam=0)[1] for s in range(5)])

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(lambdas, ridge_test, '-o', color=COLORS['train'], lw=2, ms=4, label='Ridge (λ > 0)')
ax.axhline(interp_test, color=COLORS['interp'], lw=2, ls='--', label=f'Interpolation (λ=0): {interp_test:.4f}')
ax.axhline(bayes_error, color=COLORS['bayes'], lw=1.5, ls=':', label=f'Bayes error: {bayes_error:.4f}')
ax.set_xlabel('Ridge regularization λ', fontsize=13)
ax.set_ylabel('Test MSE', fontsize=13)
ax.set_title('Interpolation vs. Ridge Regularization\n(d=1000 features, n=200 samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../assets/figures/ridge_vs_interpolation.png', dpi=120)
plt.show()

## 5. The "Raisin Bread" Intuition

Why does interpolation not catastrophically overfit?

From [Belkin et al.]:
> The interpolating predictor $f_{\text{int}}$ disagrees with the optimal $f^*$ in *basins surrounding noisy training points* — but these basins form a **set of measure zero** as $n \to \infty$.

The "raisins" are tiny isolated regions of bad prediction; the bread is correct everywhere else.

In [ ]:
# 1D illustration: interpolating a noisy signal vs. smooth fit
rng = np.random.RandomState(42)
x_1d = np.linspace(0, 1, 30)
y_clean = np.sin(2 * np.pi * x_1d)
y_noisy = y_clean + 0.25 * rng.randn(30)
# Add one very noisy point ("raisin")
y_noisy[15] += 1.5

x_dense = np.linspace(0, 1, 500)
y_true = np.sin(2 * np.pi * x_dense)

# Fit using different numbers of Fourier features
def fourier_features(x, d):
    """d sine+cosine features."""
    feats = [np.ones(len(x))]
    for k in range(1, d+1):
        feats += [np.sin(2*np.pi*k*x), np.cos(2*np.pi*k*x)]
    return np.column_stack(feats)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
configs = [(3, 'Underfit (d=3)'), (14, 'Classical sweet spot (d=14)'), (50, 'Interpolating (d=50)')]

for ax, (d_f, title) in zip(axes, configs):
    Phi_train = fourier_features(x_1d, d_f)
    Phi_dense = fourier_features(x_dense, d_f)
    alpha_f, _, _, _ = lstsq(Phi_train, y_noisy)
    y_pred = Phi_dense @ alpha_f
    
    ax.plot(x_dense, y_true, '-', color=COLORS['bayes'], lw=2, label='True f*', alpha=0.7)
    ax.plot(x_dense, y_pred, '-', color=COLORS['interp'], lw=2, label='Fitted f̂')
    ax.scatter(x_1d, y_noisy, c=COLORS['test'], s=30, zorder=5, label='Training pts')
    ax.scatter(x_1d[15], y_noisy[15], c='purple', s=100, marker='*', zorder=6, label='Noisy point')
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('The Raisin Bread: Interpolation Isolates Noise into Tiny Basins', fontsize=13)
plt.tight_layout()
plt.savefig('../assets/figures/raisin_bread.png', dpi=120)
plt.show()

## Summary

| Regime | Model size | Train error | Test error | Theory |
|--------|-----------|-------------|------------|--------|
| Underfitting | d << n | High | High | Approximation error dominates |
| Sweet spot | d ≈ n/2 | Medium | Minimum (classical) | Bias-variance balance |
| Interpolation threshold | d ≈ n | 0 | Spike (diverges) | Singular solution |
| Overparameterized | d >> n | 0 | Decreases again | Minimum-norm bias |

**Key insight:** The classical U-shaped curve ends where modern ML *starts*. Both can be correct — they describe different regimes of the same phenomenon.